# Bulk Download: Task A And Task B PCN Graphs

Bulk runner for creating desired manifests and building cached PCN graphs. Keep limits small while testing, then set them to `None` for full runs.

## 1. Configure

In [1]:
from datetime import datetime, timezone
import json

try:
    from _bootstrap import configure_paths
except ImportError:
    from notebooks._bootstrap import configure_paths

PROJECT_ROOT = configure_paths(include=("abstractgraph_graphicalizer",))

from enzyme_classification.data.protein_list import (
    exclude_enzyme_entities,
    make_human_protein_entity_manifest,
    write_manifest,
    read_manifest,
)
from enzyme_classification.pipelines.build_graphs import build_graphs_from_manifest, summarize_status

RAW_MMCIF_DIR = PROJECT_ROOT / "data/raw/mmcif"

TASK_A_ALL_MANIFEST = PROJECT_ROOT / "data/interim/task_a_all_human_proteins.tsv"
TASK_A_ENZYME_MANIFEST = PROJECT_ROOT / "data/interim/task_a_enzyme_human_proteins.tsv"
TASK_A_NON_ENZYME_MANIFEST = PROJECT_ROOT / "data/interim/task_a_non_enzyme_human_proteins.tsv"
TASK_B_MANIFEST = PROJECT_ROOT / "data/interim/task_b_enzyme_ec1_to_ec6.tsv"
TASK_A_ALL_PARTIAL_MANIFEST = PROJECT_ROOT / "data/interim/task_a_all_human_proteins.partial.tsv"
TASK_A_ENZYME_PARTIAL_MANIFEST = PROJECT_ROOT / "data/interim/task_a_enzyme_human_proteins.partial.tsv"
TASK_B_PARTIAL_MANIFEST = PROJECT_ROOT / "data/interim/task_b_enzyme_human_proteins.partial.tsv"
TASK_A_MANIFEST_META = PROJECT_ROOT / "data/interim/task_a_manifest_cache.json"
TASK_B_MANIFEST_META = PROJECT_ROOT / "data/interim/task_b_manifest_cache.json"

TASK_A_STATUS = PROJECT_ROOT / "data/processed/task_a/graph_build_status.tsv"
TASK_A_LOG = PROJECT_ROOT / "data/processed/task_a/graph_build_events.jsonl"
SHARED_GRAPH_DIR = PROJECT_ROOT / "data/processed/graphs"

TASK_B_STATUS = PROJECT_ROOT / "data/processed/task_b/graph_build_status.tsv"
TASK_B_LOG = PROJECT_ROOT / "data/processed/task_b/graph_build_events.jsonl"

MIN_DISTANCE = 4.0
MAX_DISTANCE = 8.0
SCALE_EDGE_WIDTH_BY_DISTANCE = False
MIN_EDGE_WIDTH = 0.3
MAX_EDGE_WIDTH = 2.2
OVERWRITE = False
REBUILD_MANIFESTS = False

DOWNLOAD_TIMEOUT_SECONDS = 60
DOWNLOAD_MAX_RETRIES = 4
DOWNLOAD_RETRY_BACKOFF_SECONDS = 2.0
METADATA_TIMEOUT_SECONDS = 120
METADATA_MAX_RETRIES = 5
METADATA_RETRY_BACKOFF_SECONDS = 3.0

EXPERIMENT = "full"  # "test" or "full"

EXPERIMENT_SETTINGS = {
    "test": {
        "task_a_max_entities": 100,
        "task_b_max_entities": 100,
        "page_size": 100,
        "metadata_batch_size": 50,
        "search_n_jobs": 1,
        "metadata_n_jobs": 1,
    },
    "full": {
        "task_a_max_entities": 40000,
        "task_b_max_entities": 40000,
        "page_size": 1000,
        "metadata_batch_size": 250,
        "search_n_jobs": 4,
        "metadata_n_jobs": 4,
    },
}

if EXPERIMENT not in EXPERIMENT_SETTINGS:
    raise ValueError(f"EXPERIMENT must be one of {sorted(EXPERIMENT_SETTINGS)}")

settings = EXPERIMENT_SETTINGS[EXPERIMENT]
TASK_A_MAX_ENTITIES = settings["task_a_max_entities"]
TASK_B_MAX_ENTITIES = settings["task_b_max_entities"]
PAGE_SIZE = settings["page_size"]
METADATA_BATCH_SIZE = settings["metadata_batch_size"]
SEARCH_N_JOBS = settings["search_n_jobs"]
METADATA_N_JOBS = settings["metadata_n_jobs"]

MANIFEST_CACHE_SETTINGS = {
    "manifest_builder_version": 2,
    "experiment": EXPERIMENT,
    "task_a_max_entities": TASK_A_MAX_ENTITIES,
    "task_b_max_entities": TASK_B_MAX_ENTITIES,
    "page_size": PAGE_SIZE,
    "metadata_batch_size": METADATA_BATCH_SIZE,
    "search_n_jobs": SEARCH_N_JOBS,
    "metadata_n_jobs": METADATA_N_JOBS,
}

def manifest_cache_is_current(paths, meta_path, expected_settings):
    if not all(path.exists() for path in paths):
        return False
    if not meta_path.exists():
        return False
    with meta_path.open("r", encoding="utf-8") as handle:
        metadata = json.load(handle)
    return metadata.get("settings") == expected_settings

def write_manifest_cache_metadata(meta_path, paths, settings):
    metadata = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "settings": settings,
        "paths": {path.name: str(path) for path in paths},
    }
    meta_path.parent.mkdir(parents=True, exist_ok=True)
    with meta_path.open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)

settings

{'task_a_max_entities': 40000,
 'task_b_max_entities': 40000,
 'page_size': 1000,
 'metadata_batch_size': 250,
 'search_n_jobs': 4,
 'metadata_n_jobs': 4}

## 2. Create Desired Manifests For Task A

Task A needs enzyme and non-enzyme candidates. The broad manifest contains all human protein entities returned by RCSB. The enzyme manifest contains EC-annotated entities. The non-enzyme manifest is the broad manifest minus EC-annotated entities.

In [2]:
task_a_manifest_paths = [TASK_A_ALL_MANIFEST, TASK_A_ENZYME_MANIFEST, TASK_A_NON_ENZYME_MANIFEST]
if not REBUILD_MANIFESTS and manifest_cache_is_current(task_a_manifest_paths, TASK_A_MANIFEST_META, MANIFEST_CACHE_SETTINGS):
    print("Loading cached Task A manifests")
    task_a_all = read_manifest(TASK_A_ALL_MANIFEST)
    task_a_enzyme = read_manifest(TASK_A_ENZYME_MANIFEST)
    task_a_non_enzyme = read_manifest(TASK_A_NON_ENZYME_MANIFEST)
else:
    print("Building Task A manifests from RCSB")
    task_a_all = make_human_protein_entity_manifest(
        enzyme=None,
        max_entities=TASK_A_MAX_ENTITIES,
        page_size=PAGE_SIZE,
        metadata_batch_size=METADATA_BATCH_SIZE,
        timeout_seconds=METADATA_TIMEOUT_SECONDS,
        max_retries=METADATA_MAX_RETRIES,
        retry_backoff_seconds=METADATA_RETRY_BACKOFF_SECONDS,
        incremental_cache_path=TASK_A_ALL_PARTIAL_MANIFEST,
        search_n_jobs=SEARCH_N_JOBS,
        metadata_n_jobs=METADATA_N_JOBS,
        verbose=True,
    )
    task_a_enzyme = make_human_protein_entity_manifest(
        enzyme=True,
        max_entities=TASK_A_MAX_ENTITIES,
        page_size=PAGE_SIZE,
        metadata_batch_size=METADATA_BATCH_SIZE,
        timeout_seconds=METADATA_TIMEOUT_SECONDS,
        max_retries=METADATA_MAX_RETRIES,
        retry_backoff_seconds=METADATA_RETRY_BACKOFF_SECONDS,
        incremental_cache_path=TASK_A_ENZYME_PARTIAL_MANIFEST,
        search_n_jobs=SEARCH_N_JOBS,
        metadata_n_jobs=METADATA_N_JOBS,
        verbose=True,
    )
    task_a_non_enzyme = exclude_enzyme_entities(task_a_all, task_a_enzyme)

    write_manifest(task_a_all, TASK_A_ALL_MANIFEST)
    write_manifest(task_a_enzyme, TASK_A_ENZYME_MANIFEST)
    write_manifest(task_a_non_enzyme, TASK_A_NON_ENZYME_MANIFEST)
    write_manifest_cache_metadata(TASK_A_MANIFEST_META, task_a_manifest_paths, MANIFEST_CACHE_SETTINGS)

print("Task A manifests")
print(f"all:        {len(task_a_all):>8} rows -> {TASK_A_ALL_MANIFEST}")
print(f"enzyme:     {len(task_a_enzyme):>8} rows -> {TASK_A_ENZYME_MANIFEST}")
print(f"non-enzyme: {len(task_a_non_enzyme):>8} rows -> {TASK_A_NON_ENZYME_MANIFEST}")
len(task_a_all), len(task_a_enzyme), len(task_a_non_enzyme)

Loading cached Task A manifests
Task A manifests
all:           70430 rows -> /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/interim/task_a_all_human_proteins.tsv
enzyme:        72151 rows -> /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/interim/task_a_enzyme_human_proteins.tsv
non-enzyme:    38221 rows -> /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/interim/task_a_non_enzyme_human_proteins.tsv


(70430, 72151, 38221)

In [3]:
task_a_non_enzyme.head()

,sample_id,pdb_id,label_asym_id,auth_asym_id,protein_name,entity_id,uniprot_accessions,ec_numbers,first_level_ec_numbers,resolution_angstrom
0,10en.A,10en,A,A,"Isoform 4 of Glutamate receptor ionotropic, NM...",10en_1,Q5R1P0,,,3.7
1,10en.C,10en,C,C,"Isoform 4 of Glutamate receptor ionotropic, NM...",10en_1,Q5R1P0,,,3.7
2,10en.B,10en,B,B,Glutamate receptor,10en_2,A0A8B7RCM5,,,3.7
3,10en.D,10en,D,D,Glutamate receptor,10en_2,A0A8B7RCM5,,,3.7
4,10ex.B,10ex,B,B,"Glutamate receptor ionotropic, NMDA 2B",10ex_2,Q13224,,,4.08


## 3. Create Desired Manifest For Task B

Task B uses enzyme candidates with a single first-level EC class in EC 1 through EC 6. EC 7 is excluded for paper-style reproduction.

In [4]:
task_b_manifest_paths = [TASK_B_MANIFEST]
if not REBUILD_MANIFESTS and manifest_cache_is_current(task_b_manifest_paths, TASK_B_MANIFEST_META, MANIFEST_CACHE_SETTINGS):
    print("Loading cached Task B manifest")
    task_b = read_manifest(TASK_B_MANIFEST)
else:
    print("Building Task B manifest from RCSB")
    task_b_raw = make_human_protein_entity_manifest(
        enzyme=True,
        max_entities=TASK_B_MAX_ENTITIES,
        page_size=PAGE_SIZE,
        metadata_batch_size=METADATA_BATCH_SIZE,
        timeout_seconds=METADATA_TIMEOUT_SECONDS,
        max_retries=METADATA_MAX_RETRIES,
        retry_backoff_seconds=METADATA_RETRY_BACKOFF_SECONDS,
        incremental_cache_path=TASK_B_PARTIAL_MANIFEST,
        search_n_jobs=SEARCH_N_JOBS,
        metadata_n_jobs=METADATA_N_JOBS,
        verbose=True,
    )

    single_ec = task_b_raw[task_b_raw["first_level_ec_numbers"].str.count(";") == 0].copy()
    task_b = single_ec[single_ec["first_level_ec_numbers"].isin(["1", "2", "3", "4", "5", "6"])].copy()
    write_manifest(task_b, TASK_B_MANIFEST)
    write_manifest_cache_metadata(TASK_B_MANIFEST_META, task_b_manifest_paths, MANIFEST_CACHE_SETTINGS)

print(f"Task B: {len(task_b):>8} rows -> {TASK_B_MANIFEST}")
task_b["first_level_ec_numbers"].value_counts().sort_index()

Loading cached Task B manifest
Task B:    66593 rows -> /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/interim/task_b_enzyme_ec1_to_ec6.tsv


first_level_ec_numbers
1     9753
2    23735
3    28111
4     2337
5     2050
6      607
Name: count, dtype: int64

In [5]:
task_b.head()

,sample_id,pdb_id,label_asym_id,auth_asym_id,protein_name,entity_id,uniprot_accessions,ec_numbers,first_level_ec_numbers,resolution_angstrom
0,1f0s.A,1f0s,A,A,COAGULATION FACTOR XA,1f0s_1,P00742,3;3.4;3.4.21;3.4.21.6,3,2.1
1,1f0s.B,1f0s,B,B,COAGULATION FACTOR XA,1f0s_2,P00742,3;3.4;3.4.21;3.4.21.6,3,2.1
2,1f5q.A,1f5q,A,A,CYCLIN DEPENDENT KINASE 2,1f5q_1,P24941,2;2.7;2.7.1;2.7.11;2.7.11.22,2,2.5
3,1f5q.C,1f5q,C,C,CYCLIN DEPENDENT KINASE 2,1f5q_1,P24941,2;2.7;2.7.1;2.7.11;2.7.11.22,2,2.5
4,1f8u.A,1f8u,A,A,ACETYLCHOLINESTERASE,1f8u_1,P22303,3;3.1;3.1.1;3.1.1.7,3,2.9


## 4. Bulk Build Task A Graphs

This uses the broad Task A manifest by default. For balanced development runs you can concatenate sampled enzyme and non-enzyme rows into a separate manifest before calling the builder.

In [ ]:
# Compatibility defaults for rerunning build cells in an existing notebook kernel.
SCALE_EDGE_WIDTH_BY_DISTANCE = globals().get("SCALE_EDGE_WIDTH_BY_DISTANCE", False)
MIN_EDGE_WIDTH = globals().get("MIN_EDGE_WIDTH", 0.3)
MAX_EDGE_WIDTH = globals().get("MAX_EDGE_WIDTH", 2.2)

task_a_status = build_graphs_from_manifest(
    TASK_A_ALL_MANIFEST,
    status_path=TASK_A_STATUS,
    log_path=TASK_A_LOG,
    raw_mmcif_dir=RAW_MMCIF_DIR,
    graph_dir=SHARED_GRAPH_DIR,
    min_distance=MIN_DISTANCE,
    max_distance=MAX_DISTANCE,
    scale_edge_width_by_distance=SCALE_EDGE_WIDTH_BY_DISTANCE,
    min_edge_width=MIN_EDGE_WIDTH,
    max_edge_width=MAX_EDGE_WIDTH,
    overwrite=OVERWRITE,
    download_timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    download_max_retries=DOWNLOAD_MAX_RETRIES,
    download_retry_backoff_seconds=DOWNLOAD_RETRY_BACKOFF_SECONDS,
    verbose=True,
)
summarize_status(task_a_status)

Starting graph build: 70393/70430 already done
      row       done   elapsed       eta  status        sample        pdb     name
  1/70430  70393/70430        1s   unknown  skip          10dc.A        10dc    GTPase HRas
  2/70430  70393/70430        1s   unknown  skip          10dj.A        10dj    Tyrosine-protein kinase Fyn
  3/70430  70393/70430        1s   unknown  skip          10dj.B        10dj    Tyrosine-protein kinase Fyn
  4/70430  70393/70430        1s   unknown  skip          10en.A        10en    Isoform 4 of Glutamate receptor ionotropic, NMDA 1
  5/70430  70393/70430        1s   unknown  skip          10en.C        10en    Isoform 4 of Glutamate receptor ionotropic, NMDA 1
  6/70430  70393/70430        1s   unknown  skip          10en.B        10en    Glutamate receptor
  7/70430  70393/70430        1s   unknown  skip          10en.D        10en    Glutamate receptor
  8/70430  70393/70430        1s   unknown  skip          10ex.B        10ex    Glutamate receptor ion

## 5. Bulk Build Task B Graphs

In [ ]:
# Compatibility defaults for rerunning build cells in an existing notebook kernel.
SCALE_EDGE_WIDTH_BY_DISTANCE = globals().get("SCALE_EDGE_WIDTH_BY_DISTANCE", False)
MIN_EDGE_WIDTH = globals().get("MIN_EDGE_WIDTH", 0.3)
MAX_EDGE_WIDTH = globals().get("MAX_EDGE_WIDTH", 2.2)

task_b_status = build_graphs_from_manifest(
    TASK_B_MANIFEST,
    status_path=TASK_B_STATUS,
    log_path=TASK_B_LOG,
    raw_mmcif_dir=RAW_MMCIF_DIR,
    graph_dir=SHARED_GRAPH_DIR,
    min_distance=MIN_DISTANCE,
    max_distance=MAX_DISTANCE,
    scale_edge_width_by_distance=SCALE_EDGE_WIDTH_BY_DISTANCE,
    min_edge_width=MIN_EDGE_WIDTH,
    max_edge_width=MAX_EDGE_WIDTH,
    overwrite=OVERWRITE,
    download_timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    download_max_retries=DOWNLOAD_MAX_RETRIES,
    download_retry_backoff_seconds=DOWNLOAD_RETRY_BACKOFF_SECONDS,
    verbose=True,
)
summarize_status(task_b_status)

## 6. Display Recent Logs

In [ ]:
import json

def show_recent_events(path, n=20):
    if not path.exists():
        print(f"No log yet: {path}")
        return
    events = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    for event in events[-n:]:
        print(
            f"[{event.get('ordinal', '-')}/{event.get('total', '-')}] "
            f"status={event.get('status')} done={event.get('done_count', '-')} "
            f"sample={event.get('sample_id')} name={event.get('protein_name')!r} "
            f"eta_seconds={event.get('eta_seconds')} message={event.get('message', '')}"
        )

print("Task A")
show_recent_events(TASK_A_LOG)
print("\nTask B")
show_recent_events(TASK_B_LOG)